In [ ]:
import os
import time
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Ініціалізація SparkSession (використовуємо локальні ресурси хмари Colab)
spark = SparkSession.builder \
    .appName("Lab4_Distributed_Data_Processing_Colab") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

print("Spark успішно ініціалізовано!")
print(spark)

# 2. Генерація тестового датасету (Parquet)
data_path = "mock_session_logs.parquet"
num_records = 500_000  # Зменшено до 500 тисяч для швидшої роботи в хмарі Colab

if not os.path.exists(data_path):
    print(f"\nГенерація {num_records:,} рядків симульованих логів...")
    np.random.seed(42)
    users = [f"user_{i:04d}" for i in range(1, 300)]
    sessions = [f"sess_{i:06d}" for i in range(1, 3000)]

    df_pd = pd.DataFrame({
        "user_id": np.random.choice(users, num_records),
        "session_id": np.random.choice(sessions, num_records),
        "event_timestamp": np.random.randint(1710000000000, 1710003600000, num_records, dtype=np.int64)
    })

    spark_df_temp = spark.createDataFrame(df_pd)
    spark_df_temp.write.mode("overwrite").parquet(data_path)
    print("Датасет успішно записано у формат Parquet.")

# Читання даних із джерела
df = spark.read.parquet(data_path)


# 3. Стратегія А: «Високе перетасування» (Наївна)
print("\n" + "="*50)
print("Виконання Стратегії А ('Високе перетасування')...")
print("="*50)

start_time_a = time.time()

# Примусове завищення кількості партицій для генерації інтенсивного shuffle
df_strategy_a = df.repartition(200, "user_id")

session_durations_a = df_strategy_a.groupBy("user_id", "session_id") \
    .agg((F.max("event_timestamp") - F.min("event_timestamp")).alias("duration_ms"))

avg_duration_a = session_durations_a.groupBy().agg(F.avg("duration_ms")).collect()[0][0]
time_a = time.time() - start_time_a

print(f"Час виконання Стратегії А: {time_a:.4f} сек.")


# 4. Стратегія В: «Низьке перетасування» (Оптимізована)
print("\n" + "="*50)
print("Виконання Стратегії В ('Низьке перетасування')...")
print("="*50)

start_time_b = time.time()

# Оптимізація: використовуємо стандартний парсинг без зайвого примусового repartition
spark.conf.set("spark.sql.shuffle.partitions", "4")

session_durations_b = df.groupBy("user_id", "session_id") \
    .agg((F.max("event_timestamp") - F.min("event_timestamp")).alias("duration_ms"))

avg_duration_b = session_durations_b.groupBy().agg(F.avg("duration_ms")).collect()[0][0]
time_b = time.time() - start_time_b

print(f"Час виконання Стратегії В: {time_b:.4f} сек.")


# 5. Підсумки та аналіз
print("\n" + "="*50)
print("ПІДСУМКОВІ РЕЗУЛЬТАТИ ДЛЯ ЗВІТУ")
print("="*50)
speedup = time_a / time_b if time_b > 0 else 1.0
print(f"Прискорення (Strategy A time / Strategy B time): {speedup:.2f}x")
print(f"Середня тривалість сеансу (А): {avg_duration_a:.2f} мс")
print(f"Середня тривалість сеансу (В): {avg_duration_b:.2f} мс")

# Завершення сесії Spark
spark.stop()

Spark успішно ініціалізовано!

Генерація 500,000 рядків симульованих логів...
Датасет успішно записано у формат Parquet.

Виконання Стратегії А ('Високе перетасування')...
Час виконання Стратегії А: 7.5278 сек.

Виконання Стратегії В ('Низьке перетасування')...
Час виконання Стратегії В: 2.4970 сек.

ПІДСУМКОВІ РЕЗУЛЬТАТИ ДЛЯ ЗВІТУ
Прискорення (Strategy A time / Strategy B time): 3.01x
Середня тривалість сеансу (А): 332265.95 мс
Середня тривалість сеансу (В): 332265.95 мс
